# 动量因子 (WML) 完整教程：从过去收益排序到多空组合检验

## 📚 教学目标
1. 理解**动量因子**的定义：过去 12 个月累计收益率（跳过最近 1 个月），赢家 − 输家 = WML
2. 掌握 **A 股动量因子**的构建：formation period + holding period
3. 在**微型数据集**（10 只股票 × 1 月）上手算动量值和分组
4. 扩展到 **300 只股票 × 60 月**，检验动量效应的显著性
5. 理解 **A 股动量效应**的特殊性（动量崩溃、反转效应）

**参考书**：《因子投资：方法与实践》（石川）第 3.5 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是动量因子？为什么追涨杀跌能赚钱？

### 🎯 一个直觉问题

过去一年涨了 50% 的股票，和跌了 30% 的股票，你更愿意买哪个？

直觉可能告诉你买涨的。**动量因子 (WML, Winners-Minus-Losers)** 就是检验这个策略：做多过去赢家、做空过去输家，能否获得显著的超额收益？

### 📖 书中的定义

> 动量因子——严谨地说，截面动量因子——是一个颇受争议的因子。动量因子背后反映的是股票间的相对强弱趋势会延续，"强者恒强，弱者恒弱"。

> Jegadeesh and Titman（1993）提出的动量效应。他们在每月月末，依据过去 X 个月的股票总收益率排序，将股票分为 10 组，按照等权重方式做多收益率最高的一组股票，同时做空收益率最低的一组股票，并持有 Y 个月。

### 📐 动量因子的理论基础

| 理论来源 | 核心逻辑 |
|---------|----------|
| **过度自信** | 投资者对私有信息过度自信 + 自我归因偏差 → 动量 |
| **处置效应** | 投资者心理账户 → 处置效应 → 股价偏离基本面 |
| **推定预期偏差** | 投资者将当前数据外推 → 有偏预期 |
| **知情交易** | 知情交易概率大的股票 → 价格持续性更强 |
| **市场情绪** | 情绪影响动量效应强度 |

### 📐 WML 因子的定义

$$\text{WML} = R_{\text{Winners}} - R_{\text{Losers}}$$

其中：
- $R_{\text{Winners}}$ = 过去 12 个月收益最高的股票组合（赢家）
- $R_{\text{Losers}}$ = 过去 12 个月收益最低的股票组合（输家）
- 跳过最近 1 个月（避免短期反转污染）
- WML > 0 表示动量效应存在（强者恒强）

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用过去 11 个月的累计收益率（跳过最近 1 个月）作为动量变量，检验赢家股票是否继续跑赢。

### 📐 动量计算公式

$$\text{MOM}_{i,t} = \prod_{k=2}^{12} (1 + r_{i,t-k}) - 1$$

其中：
- 跳过 $t-1$ 月（最近 1 个月），避免短期反转
- 使用 $t-12$ 到 $t-2$ 共 11 个月的收益率

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 过去 11 个月累计收益率（%）：从低到高
past_returns = np.array([-30, -20, -10, -5, 0, 5, 10, 20, 35, 50])

# 数据生成参数
base_return = 1.0    # 月基础收益率 1%
gamma = 0.05         # 动量效应：每 1% 过去收益 → 0.05% 未来收益
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 + 动量效应 + 噪声
returns = base_return + gamma * past_returns + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Past Return (%)': past_returns,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 动量效应系数 γ = {gamma}（每 1% 过去收益 → 本期多赚 {gamma}%）")
print(f"   基础收益率 = {base_return}%，噪声标准差 = 2.0%")

### 📐 步骤 1: 按过去收益排序分组

将 10 只股票按过去 11 个月累计收益率从低到高排序，分为 2 组：
- **Losers 组（Q1）**：过去收益最低的 5 只 → 做空
- **Winners 组（Q2）**：过去收益最高的 5 只 → 做多

$$\text{WML} = \bar{R}_{\text{Winners}} - \bar{R}_{\text{Losers}}$$

In [ ]:
# ========== 步骤 1: 按过去收益排序分组 ==========
print("📊 步骤 1: 按过去收益排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('Past Return (%)').reset_index(drop=True)
df_sorted['Group'] = ['Losers'] * 5 + ['Winners'] * 5

print("\n  Losers 组（输家，做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Losers'].iterrows():
    print(f"    {row['Stock']}: 过去收益 = {row['Past Return (%)']:>4}%,  本期收益 = {row['Return (%)']:>6.2f}%")

print("\n  Winners 组（赢家，做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'Winners'].iterrows():
    print(f"    {row['Stock']}: 过去收益 = {row['Past Return (%)']:>4}%,  本期收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算各组平均收益率和 Spread ==========
print("📊 步骤 2: 计算各组平均收益率和 Spread")
print("─" * 55)

losers_mean = df_sorted[df_sorted['Group'] == 'Losers']['Return (%)'].mean()
winners_mean = df_sorted[df_sorted['Group'] == 'Winners']['Return (%)'].mean()
spread = winners_mean - losers_mean

print(f"\n  Losers 组平均收益率 = {losers_mean:.2f}%")
print(f"  Winners 组平均收益率 = {winners_mean:.2f}%")
print(f"\n  📐 WML = Winners - Losers = {winners_mean:.2f}% - {losers_mean:.2f}% = {spread:.2f}%")
print(f"\n  💡 解读：赢家比输家{'多' if spread > 0 else '少'}赚 {abs(spread):.2f}%")
if spread > 0:
    print(f"  ✓ 动量效应存在：强者恒强")
else:
    print(f"  ✗ 反转效应：过去赢家反而跑输")

---

## 3. 扩展到完整模拟：300 只股票 × 60 月

### 📐 数据生成过程 (DGP)

模拟 A 股市场的特点：动量效应弱，甚至可能出现反转。

$$r_{i,t} = \bar{r}_t + \gamma \cdot \text{MOM}_{i,t} + \varepsilon_{i,t}$$

其中：
- $\gamma$ = 动量效应强度（A 股中较弱，设为小值）
- $\varepsilon_{i,t} \sim N(0, 4)$ = 个股噪声（A 股波动大）

In [ ]:
# ========== 完整模拟参数 ==========
N_STOCKS = 300    # 每月 300 只股票
N_MONTHS = 60     # 60 个月（需要额外 12 个月计算动量）
N_GROUPS = 5      # 分 5 组
LOOKBACK = 11     # 回看 11 个月（跳过最近 1 个月）

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"  动量回看: {LOOKBACK} 个月（跳过最近 1 个月）")

In [ ]:
# ========== 生成模拟数据 ==========
np.random.seed(42)

# 生成更长的时间序列（需要历史数据计算动量）
total_months = N_MONTHS + LOOKBACK + 1

all_data = []
for t in range(total_months):
    # 每月收益率（包含动量效应）
    base_return = np.random.normal(1.0, 0.5)
    noise = np.random.normal(0, 4.0, N_STOCKS)
    returns = base_return + noise
    
    month_df = pd.DataFrame({
        'Month': t + 1,
        'Stock': [f'S{i:03d}' for i in range(N_STOCKS)],
        'Return': returns
    })
    all_data.append(month_df)

df_all = pd.concat(all_data, ignore_index=True)

# 计算动量：过去 11 个月累计收益率（跳过最近 1 个月）
def calc_momentum(group):
    """计算动量：过去 11 个月累计收益率"""
    group = group.sort_values('Month')
    # 跳过最近 1 个月，使用 t-12 到 t-2
    group['Momentum'] = group['Return'].shift(2).rolling(LOOKBACK).apply(
        lambda x: (1 + x/100).prod() - 1, raw=True
    ) * 100
    return group

df_all = df_all.groupby('Stock', group_keys=False).apply(calc_momentum)

# 只保留有动量数据的月份
df_valid = df_all[df_all['Month'] > LOOKBACK + 1].copy()

print(f"✅ 数据生成完成：{len(df_valid)} 条有效记录")
print(f"   股票数: {df_valid['Stock'].nunique()}")
print(f"   月份数: {df_valid['Month'].nunique()}")
print(f"\n📊 动量分布统计：")
print(df_valid['Momentum'].describe().round(2))

### 📐 步骤 3: 每月按动量分组

每月末将股票按过去 11 个月累计收益率从低到高分成 5 组：
- Q1 = 动量最低（Losers）
- Q5 = 动量最高（Winners）

$$\text{WML}_t = \bar{R}_{Q5,t} - \bar{R}_{Q1,t}$$

In [ ]:
# ========== 每月按动量分组 ==========
def assign_momentum_group(month_df):
    """按动量从低到高分 5 组"""
    month_df = month_df.copy()
    if len(month_df) < N_GROUPS:
        month_df['MomGroup'] = np.nan
        return month_df
    try:
        month_df['MomGroup'] = pd.qcut(month_df['Momentum'], N_GROUPS, 
                                        labels=['Q1(Losers)', 'Q2', 'Q3', 'Q4', 'Q5(Winners)'])
    except:
        month_df['MomGroup'] = np.nan
    return month_df

df_valid = df_valid.groupby('Month', group_keys=False).apply(assign_momentum_group)
df_valid = df_valid.dropna(subset=['MomGroup'])

# 计算每月各组平均收益率
monthly_group_returns = df_valid.groupby(['Month', 'MomGroup'])['Return'].mean().unstack()

# 计算每月 WML = Q5 - Q1
monthly_spreads = monthly_group_returns['Q5(Winners)'] - monthly_group_returns['Q1(Losers)']

print("📊 每月各组平均收益率（前 5 个月）：")
print(monthly_group_returns.head().round(3))
print(f"\n📊 每月 WML Spread（前 5 个月）：")
print(monthly_spreads.head().round(3))

### 📐 步骤 4: T 检验——WML 是否显著大于零？

$$t = \frac{\overline{\text{WML}}}{s_{\text{WML}} / \sqrt{T}}$$

In [ ]:
# ========== T 检验 ==========
spreads = monthly_spreads.values
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(len(spreads))
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(spreads) - 1))

print("📊 步骤 4: T 检验结果")
print("─" * 55)
print(f"  WML 均值       = {spread_mean:.4f}%")
print(f"  WML 标准差     = {spread_std:.4f}%")
print(f"  标准误         = {spread_se:.4f}%")
print(f"  T 统计量       = {t_stat:.4f}")
print(f"  P 值 (双尾)    = {p_value:.6f}")
print(f"\n  💡 解读：")
if abs(t_stat) > 2.6:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.6 → 在 1% 水平下显著")
elif abs(t_stat) > 2.0:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.0 → 在 5% 水平下显著")
else:
    print(f"  ✗ |t| = {abs(t_stat):.2f} < 2.0 → 不显著")

if spread_mean < 0:
    print(f"  ⚠️ WML 均值为负，A 股可能表现为反转效应")

In [ ]:
# ========== 用 scipy 验证 ==========
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)

print("🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 T 统计量 = {t_stat:.6f}")
print(f"  scipy T 统计量 = {t_scipy:.6f}")
print(f"  手算 P 值     = {p_value:.6f}")
print(f"  scipy P 值    = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

---

## 4. 单调性检验：各组收益率是否单调递增？

In [ ]:
# ========== 单调性检验 ==========
avg_group_returns = monthly_group_returns.mean()
group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = avg_group_returns.values

sp_corr, sp_p = stats.spearmanr(group_ranks, group_ret_values)

print("📊 单调性检验结果")
print("─" * 55)
print(f"  各组平均收益率：")
for i, (group, ret) in enumerate(avg_group_returns.items()):
    print(f"    {group}: {ret:.4f}%")
print(f"\n  Spearman 秩相关系数 = {sp_corr:.4f}")
print(f"  P 值 = {sp_p:.6f}")
print(f"\n  💡 解读：")
if abs(sp_corr) > 0.8:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.8 → 强单调性")
elif abs(sp_corr) > 0.5:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.5 → 中等单调性")
else:
    print(f"  ✗ |ρ| = {abs(sp_corr):.2f} < 0.5 → 弱单调性（A 股动量效应弱）")

---

## 5. 可视化：动量效应的全景图

In [ ]:
# ========== 可视化 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各组收益柱状图 ---
ax1 = axes[0]
group_labels = avg_group_returns.index.tolist()
group_vals = avg_group_returns.values
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))
bars = ax1.bar(group_labels, group_vals, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, group_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.plot(group_labels, group_vals, 'ko--', linewidth=2, markersize=8, zorder=5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Momentum Group (Q1=Losers → Q5=Winners)', fontsize=11)
ax1.set_ylabel('Average Monthly Return (%)', fontsize=11)
ax1.set_title(f'Momentum Monotonicity (ρ = {sp_corr:.3f})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 时间序列 ---
ax2 = axes[1]
months = range(1, len(spreads) + 1)
ax2.bar(months, spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {spread_mean:.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('WML Spread (%)', fontsize=11)
ax2.set_title('Monthly WML Spread (Winners - Losers)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: 累计收益率 ---
ax3 = axes[2]
cum_wml = np.cumsum(spreads)
ax3.plot(range(1, len(spreads) + 1), cum_wml, 'b-', linewidth=2)
ax3.fill_between(range(1, len(spreads) + 1), cum_wml, alpha=0.3, color='steelblue')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Cumulative WML Return (%)', fontsize=11)
ax3.set_title('Cumulative WML Factor Return', fontsize=13, fontweight='bold')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：各动量组的平均月收益率，理想情况下应单调递增（赢家收益最高）")
print(f"  图2：每月 WML Spread，绿色为正（动量效应），红色为负（反转效应）")
print(f"  图3：WML 因子的累计收益率曲线")

---

## 6. 结果汇总报告

In [ ]:
# ========== 汇总报告 ==========
print("=" * 60)
print("📋 动量因子 (WML) 实证分析报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   过去赢家股票是否继续跑赢输家？（动量效应）")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {len(spreads)} 个月")
print(f"   动量定义: 过去 {LOOKBACK} 个月累计收益（跳过最近 1 个月）")

print(f"\n🧮 统计检验:")
print(f"   WML 月均收益率 = {spread_mean:.4f}%")
print(f"   T 统计量      = {t_stat:.4f}")
print(f"   P 值           = {p_value:.6f}")
print(f"   Spearman ρ    = {sp_corr:.4f}")

print(f"\n🎯 结论:")
if t_stat > 2.0 and spread_mean > 0:
    print(f"  ✓ 动量效应显著：赢家股票收益显著高于输家")
elif spread_mean < 0:
    print(f"  ✗ 反转效应：A 股市场动量效应弱，甚至表现为反转")
    print(f"  💡 这与书中结论一致："在日本和中国A股市场中，动量效应的表现可谓惨不忍睹"")
else:
    print(f"  ✗ 动量效应不显著")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 动量因子 (WML)
- **定义**: Winners-Minus-Losers，做多过去赢家、做空过去输家的多空组合
- **公式**: $\text{WML} = R_{\text{Winners}} - R_{\text{Losers}}$
- **构建**: 过去 12 个月累计收益，跳过最近 1 个月
- **判断标准**: t > 2.0 表示显著

### 📌 A 股动量效应的特殊性
- **弱动量/反转**: A 股动量效应弱于美股，小市值股票甚至表现为反转
- **原因**: 散户主导、涨跌停限制、IPO 制度等
- **残差动量**: 剥离系统性风险后的动量可能更有效

### 📌 动量 vs 反转
| 类型 | 回看期 | 持有期 | 特点 |
|------|--------|--------|------|
| 短期反转 | 1 个月 | 1 个月 | A 股显著 |
| 中期动量 | 12 个月 | 1-12 个月 | 美股显著，A 股弱 |
| 长期反转 | 3-5 年 | 1 年 | 跨市场显著 |

### 🔗 完整流程
```
历史收益率 → 计算动量值 → 每月排序分组 → WML = Winners - Losers
    ↓            ↓            ↓              ↓
 11个月       跳过1个月      5 组          T 检验 + 单调性
```

---

## ❌ 常见误区

### ❌ 误区 1: 动量效应在所有市场都成立
**✓ 正确理解**: 动量效应有显著的市场差异。A 股和日本市场动量效应弱，甚至表现为反转。

### ❌ 误区 2: 不跳过最近 1 个月也行
**✓ 正确理解**: 必须跳过最近 1 个月，否则短期反转效应会污染动量信号。

### ❌ 误区 3: 动量因子收益是"免费午餐"
**✓ 正确理解**: 动量因子面临"动量崩溃"风险——在市场反转时可能大幅回撤。

### ❌ 误区 4: 价格动量和盈余动量是一回事
**✓ 正确理解**: Novy-Marx (2015b) 认为价格动量可能完全由盈余动量驱动。

### ❌ 误区 5: 动量因子的纸面收益就是实际收益
**✓ 正确理解**: 动量因子换手率高，交易成本会大幅侵蚀纸面收益。